# FinBERT Fine-tuning — Tone Classification
**Run this notebook on GPU: Runtime → Change runtime type → T4 GPU**

Pipeline:
1. Install dependencies
2. Upload `labeled_chunks.parquet` from your machine
3. Fine-tune FinBERT (≈ 20 min on T4)
4. Download the saved model


In [ ]:
# ── Step 1: Check GPU ─────────────────────────────────────────────────────────
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — switch to GPU!'}")

In [ ]:
# ── Step 2: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers scikit-learn pyarrow pandas

In [ ]:
# ── Step 3: Upload labeled_chunks.parquet ─────────────────────────────────────
from google.colab import files
print("Upload your labeled_chunks.parquet file:")
uploaded = files.upload()

In [ ]:
# ── Step 4: Load and inspect data ─────────────────────────────────────────────
import pandas as pd

df = pd.read_parquet("labeled_chunks.parquet")
print(f"Total labeled chunks: {len(df)}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nConfidence stats:")
print(df['confidence'].describe().round(3))

# Drop low-confidence labels
df = df[df['confidence'] >= 0.6].copy()
print(f"\nAfter confidence filter: {len(df)} chunks")

In [ ]:
# ── Step 5: Train/val/test split ──────────────────────────────────────────────
from sklearn.model_selection import train_test_split

LABEL2ID = {"optimistic": 0, "cautious": 1, "negative": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

df["label_id"] = df["label"].map(LABEL2ID)
df = df.dropna(subset=["label_id"])

texts  = df["content"].tolist()
labels = df["label_id"].astype(int).tolist()

X_train, X_tmp, y_train, y_tmp = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42
)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
# ── Step 6: Dataset class ─────────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class ToneDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True,
            max_length=512, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

train_ds = ToneDataset(X_train, y_train, tokenizer)
val_ds   = ToneDataset(X_val,   y_val,   tokenizer)
test_ds  = ToneDataset(X_test,  y_test,  tokenizer)

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)
print("Datasets ready.")

In [ ]:
# ── Step 7: Load FinBERT + add classification head ────────────────────────────
# Architecture (Session 3):
#   FinBERT backbone -> [CLS] 768-dim -> Linear(768, 3) -> Softmax
#   Loss: CrossEntropy (exactly one label per chunk)
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
model.to(device)
print(f"Model loaded on {device}")

EPOCHS      = 3
LR          = 2e-5
optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

In [ ]:
# ── Step 8: Training loop ─────────────────────────────────────────────────────
import numpy as np

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    total_loss, preds, true = 0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds.extend(outputs.logits.argmax(dim=-1).cpu().tolist())
            true.extend(labels.cpu().tolist())
    return total_loss / len(loader), preds, true

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_preds, val_labels = evaluate(model, val_loader, device)
    val_acc = np.mean(np.array(val_preds) == np.array(val_labels))
    print(f"Epoch {epoch}/{EPOCHS} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained("/content/finbert_tone")
        tokenizer.save_pretrained("/content/finbert_tone")
        print(f"  -> Best model saved")

In [ ]:
# ── Step 9: Test set evaluation ───────────────────────────────────────────────
from sklearn.metrics import classification_report

_, test_preds, test_labels = evaluate(model, test_loader, device)
report = classification_report(
    test_labels, test_preds,
    target_names=["optimistic", "cautious", "negative"]
)
print(report)

# Save report
with open("/content/eval_report.txt", "w") as f:
    f.write(report)
print("Report saved.")

In [ ]:
# ── Step 10: Download model ───────────────────────────────────────────────────
import shutil
from google.colab import files

# Zip the model folder
shutil.make_archive("/content/finbert_tone", "zip", "/content/finbert_tone")

# Download both files
files.download("/content/finbert_tone.zip")
files.download("/content/eval_report.txt")
print("Download started. Unzip finbert_tone.zip into models/finbert_tone/ on your machine.")